🧠 LangChain (Agent & Chain Playground)

* langchain – main framework
* langchain-community – loaders, tools, integrations
* langchain-core – prompts, runnables, LCEL
* langchain-text-splitters – chunking documents

🤗 Hugging Face + Transformers

* transformers – models, pipelines
* huggingface-hub – model downloads & auth
* datasets – datasets & evaluation
* sentencepiece,
* tokenizers – tokenizer backends

⚡ Performance & Fine-Tuning

* torch – backbone (Kaggle GPU already supports it)
* accelerate – multi-GPU / mixed precision
* bitsandbytes – 8-bit / 4-bit loading
* peft – LoRA, adapters
* einops – tensor reshaping sugar

In [1]:
!pip install \
    langchain \
    langchain-huggingface \
    langchain-community \
    transformers \
    accelerate \
    bitsandbytes \
    sentence-transformers

INFO: pip is looking at multiple versions of langchain-huggingface to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of langchain-community to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 30.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 29.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 3.5 MB/s eta 0:00:00
  Attempting uninstall: packaging
    Found existing installation: packaging 26.0rc2
    Uninstalling packaging-26.0rc2:
      Successfully uninstalled packaging-26.0rc2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.22.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installe

# import required stuff

In [3]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

login(token=hf_token)

In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, AutoModel, AutoTokenizer
from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
from langchain_huggingface import HuggingFacePipeline

print("Transformers + LangChain ready ✅")
print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())


2026-02-04 17:07:40.835099: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770224861.304183      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770224861.428839      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770224862.592442      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770224862.592521      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770224862.592524      55 computation_placer.cc:177] computation placer alr

Transformers + LangChain ready ✅
Torch: 2.8.0+cu126
CUDA: True


# ----------------------------------------------------------

# ibm-granite/granite-3.0-2b-instruct

# ---------------------------------------------------------

In [5]:
# --- အပိုင်း (၁): Model ကို GPU ပေါ်မှာ တစ်ခါတည်း တင်ထားခြင်း ---
model_id = "ibm-granite/granite-3.0-2b-instruct"

#model_id = "meta-llama/Llama-3.1-8B-Instruct"

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Load Model and Related Parameters
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/87.0 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [6]:
# --- အပိုင်း (၂): Parameters တွေကို ပြောင်းပြီး Invoke လုပ်မယ့် Function ---
def llm_model(prompt_txt, user_params=None):
    # Default parameters
    params = {
        "max_new_tokens": 256,
        "temperature": 0.5,
        "top_p": 0.2,
        "top_k": 1,
    }
    
    # User က params အသစ်ပေးလာရင် overwrite လုပ်မယ်
    if user_params:
        params.update(user_params)
    
    pipe = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        **params,           # params အသစ်ပေးလာရင် overwrite လုပ်မယ်
        return_full_text=False
    )
    
    # LangChain wrap လုပ်ပြီး invoke လုပ်မယ်
    llm = HuggingFacePipeline(pipeline=pipe)
    return llm.invoke(prompt_txt)


# Params ပြောင်းလဲပြီးအသုံးပြုလို့ရတဲ့ ပုံစံလေးကိုစမ်းသပ်ကြည့်ရုံပါ။

In [7]:
# --- အပိုင်း (၃): မတူညီတဲ့ Task တွေအတွက် စမ်းသပ်ပုံ ---

# Task 1: စာတိုလေးပဲ လိုချင်တာ (Short generation)
params_1 = {
    "max_new_tokens": 20,
    "temperature": 0.1 # ပိုပြီး တိကျစေချင်လို့ (Creative မဖြစ်စေချင်လို့)
}
print(f"Task 1 Response: {llm_model('The wind is', params_1)}\n")

# Task 2: စိတ်ကူးယဉ်စာသား အရှည်ကြီး လိုချင်တာ (Creative & Long generation)
params_2 = {
    "max_new_tokens": 128,
    "min_new_tokens": 10,
    "temperature": 0.9, # ပိုပြီး Creative ဖြစ်စေဖို့
    "top_p": 0.9
}
print(f"Task 2 Response: {llm_model('The wind is', params_2)}")

Device set to use cuda:0
Device set to use cuda:0


Task 1 Response:  blowing at 20 mph, and the temperature is 70 degrees Fahren

Task 2 Response:  blowing at 20 mph, and the temperature is 70 degrees Fahrenheit. If you are standing still on the ground, how long would it take for the wind to blow you over?

To answer this question, we need to consider the forces acting on you and the time it takes for those forces to change your direction. The wind's force is given by the equation F = 0.5 * ρ * A * Cd * v^2, where ρ is the air density (approximately 1.225 kg/m^3 at sea level


# ---------------------------------------------------------------------

# Generation Example

In [8]:
params = {
    "max_new_tokens": 128,
    "min_new_tokens": 10,
    "temperature": 1.2, 
    "top_p": 0.9
    # "do_sample": True,
    # "repetition_penalty": 1.05
}

prompt = """The wind is"""

# Getting a reponse from the model with the provided prompt and new parameters

print(f"Task 2 Response: {llm_model(prompt, params)}")

Device set to use cuda:0


Task 2 Response:  blowing at 20 mph, and the temperature is 70 degrees Fahrenheit. If you are standing still on the ground, how long would it take for the wind to blow you over?

To answer this question, we need to consider the forces acting on you and the time it takes for those forces to change your direction. The wind's force is given by the equation F = 0.5 * ρ * A * Cd * v^2, where ρ is the air density (approximately 1.225 kg/m^3 at sea level


# ------------------------------------------------------------------

## zeroshot Classification

In [9]:
prompt = """Classify the following statement as true or false: 
            'The Eiffel Tower is located in Berlin.'

            Answer:
"""
response = llm_model(prompt, params)
print(f"prompt: {prompt}\n")
print(f"response : {response}\n")

Device set to use cuda:0


prompt: Classify the following statement as true or false: 
            'The Eiffel Tower is located in Berlin.'

            Answer:


response : 
False. The Eiffel Tower is located in Paris, France.



## One-shot: Classification


In [10]:
params = {
    "max_new_tokens": 20,
    "temperature": 0.1,
}

prompt = """Here is an example of translating a sentence from English to French:

            English: “How is the weather today?”
            French: “Comment est le temps aujourd'hui?”
            
            Now, translate the following sentence from English to French:
            
            English: “Where is the nearest supermarket?”
            
"""
response = llm_model(prompt, params)
print(f"prompt: {prompt}\n")
print(f"response : {response}\n")

Device set to use cuda:0


prompt: Here is an example of translating a sentence from English to French:

            English: “How is the weather today?”
            French: “Comment est le temps aujourd'hui?”
            
            Now, translate the following sentence from English to French:
            
            English: “Where is the nearest supermarket?”
            


response : 
French: "Où est le supermarché le plus proche?"



## Few Shot Classification

In [11]:
# parameters: Set `max_new_tokens` to 10, which constrains the model to generate brief responses

params = {
    "max_new_tokens": 100,
}

prompt = """Here are few examples of classifying emotions in statements:

            Statement: 'I just won my first marathon!'
            Emotion: Joy
            
            Statement: 'I can't believe I lost my keys again.'
            Emotion: Frustration
            
            Statement: 'My best friend is moving to another country.'
            Emotion: Sadness
            
            Now, classify the emotion in the following statement:
            Statement: 'That movie was so scary I had to cover my eyes.’
            

"""
response = llm_model(prompt, params)
print(f"prompt: {prompt}\n")
print(f"response : {response}\n")

Device set to use cuda:0


prompt: Here are few examples of classifying emotions in statements:

            Statement: 'I just won my first marathon!'
            Emotion: Joy
            
            Statement: 'I can't believe I lost my keys again.'
            Emotion: Frustration
            
            Statement: 'My best friend is moving to another country.'
            Emotion: Sadness
            
            Now, classify the emotion in the following statement:
            Statement: 'That movie was so scary I had to cover my eyes.’
            



response : 
The emotion in the statement "That movie was so scary I had to cover my eyes" is Fear.



## Chain-of-Thought Prompting

In [16]:
params = {
    "max_new_tokens": 128,
    "min_new_tokens" : 28,
    "temperature": 0.5,
}

prompt = """Consider the problem: 'A store had 22 apples. They sold 15 apples today and got a new delivery of 8 apples. 
            How many apples are there now?’

            Break down each step of your calculation

"""
response = llm_model(prompt, params)
print(f"prompt: {prompt}\n")
print(f"response : {response}\n")

Device set to use cuda:0


prompt: Consider the problem: 'A store had 22 apples. They sold 15 apples today and got a new delivery of 8 apples. 
            How many apples are there now?’

            Break down each step of your calculation



response : 
1. Start with the initial number of apples: 22
2. Subtract the number of apples sold: 22 - 15 = 7
3. Add the number of apples received in the delivery: 7 + 8 = 15

So, there are now 15 apples in the store.



## decision-making process
## explaining a process

In [13]:
params = {
    "max_new_tokens": 512,
    "min_new_tokens" : 256,
    "temperature": 1,
}

# 1. Prompt for decision-making process
decision_making_prompt = """
Consider this situation: A student is trying to decide whether to study tonight or go to a movie with friends. They have a test in two days.

Think through this decision step-by-step, considering the pros and cons of each option, and what factors might be most important in making this choice.
"""

# 2. Prompt for explaining a process
sandwich_making_prompt = """
Explain how to make a peanut butter and jelly sandwich.

Break down each step of the process in detail, from gathering ingredients to finishing the sandwich.
"""

responses = {}
responses["decision_making"] = llm_model(decision_making_prompt)
responses["sandwich_making"] = llm_model(sandwich_making_prompt)

for prompt_type, response in responses.items():
    print(f"=== {prompt_type.upper()} RESPONSE ===")
    print(response)
    print()

Device set to use cuda:0
Device set to use cuda:0


=== DECISION_MAKING RESPONSE ===

Studying tonight:

Pros:

* The student will be better prepared for the test, which is in two days.
* The student can review material that they may have missed or not understood fully.
* The student can get a head start on their studying, which may help reduce stress and anxiety leading up to the test.

Cons:

* The student may feel tired or unmotivated after a long day of classes and work.
* The student may feel like they are missing out on social activities with their friends.
* The student may feel like they are not getting enough downtime or relaxation.

Going to the movie with friends:

Pros:

* The student can relax and have fun with their friends.
* The student can take a break from studying and give their mind a rest.
* The student may feel more motivated and energized to study the next day after a night of relaxation.

Cons:

* The student may feel like they are procrastinating and putting off their studies.
* The student may feel like they ar

In [17]:
params = {
    "max_new_tokens": 512,
}

prompt = """When I was 6, my sister was half of my age. Now I am 70, what age is my sister?

            Provide three independent calculations and explanations, then determine the most consistent result.

"""
response = llm_model(prompt, params)
print(f"prompt: {prompt}\n")
print(f"response : {response}\n")

Device set to use cuda:0


KeyboardInterrupt: 

## Introduction to LangChain 
1. Define the content or problem to be addressed.
2. Create a template with variables for dynamic content.
3. Convert the template into a LangChain PromptTemplate.
4. Build a chain using the pipe operator `|` to connect:

    - Input variables
    - The prompt template
    - The LLM
    - An output parser

# **Next Section 02_LangChain Framework**